In [10]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings

warnings.filterwarnings("ignore")

In [11]:
df = pd.read_csv("../Data/disease_dataset.csv")
print("🚀 High-accuracy dataset loaded!")
print(f"Shape: {df.shape}")
print("\nDisease distribution:")
print(df["prognosis"].value_counts().sort_index())
print("\nDataset preview:")
df.head()

🚀 High-accuracy dataset loaded!
Shape: (5000, 17)

Disease distribution:
prognosis
Allergy            500
Arthritis          500
Asthma             500
Common Cold        500
Flu                500
Food Poisoning     500
Gastroenteritis    500
Hypertension       500
Migraine           500
Pneumonia          500
Name: count, dtype: int64

Dataset preview:


,prognosis,fever,cough,headache,fatigue,sneezing,runny_nose,nausea,stomach_pain,shortness_breath,chest_pain,high_bp,dizziness,joint_pain,itching,vomiting,diarrhea
0,Common Cold,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0
1,Common Cold,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0
2,Common Cold,1,1,0,1,1,1,0,0,0,0,0,0,0,0,0,1
3,Common Cold,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0
4,Common Cold,1,1,1,1,1,1,0,0,0,0,0,0,1,0,0,0


In [12]:
symptom_cols = [
    "fever",
    "cough",
    "headache",
    "fatigue",
    "nausea",
    "stomach_pain",
    "shortness_breath",
    "chest_pain",
    "sneezing",
    "runny_nose",
    "high_bp",
    "dizziness",
    "joint_pain",
    "itching",
    "vomiting",
    "diarrhea",
]

X = df[symptom_cols]
y = df["prognosis"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"✅ Training: {X_train.shape}, Test: {X_test.shape}")
print(f"Diseases: {len(le.classes_)}")

✅ Training: (4000, 16), Test: (1000, 16)
Diseases: 10


In [13]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=10),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=None, random_state=42
    ),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=2000, C=1.0),
    "Bernoulli NB": BernoulliNB(alpha=0.01, binarize=0.0),
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"✅ {name} trained!")

print("\n🎯 All models ready for predictions!")

✅ Decision Tree trained!
✅ Random Forest trained!
✅ Logistic Regression trained!
✅ Bernoulli NB trained!

🎯 All models ready for predictions!


In [14]:
results = {}
detailed_reports = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    detailed_reports[name] = classification_report(
        y_test, y_pred, target_names=le.classes_, output_dict=True
    )

    print(f"📊 {name}: **{acc:.4f}** ({acc*100:.2f}%)")
    print(f"   Precision: {detailed_reports[name]['weighted avg']['precision']:.4f}")
    print(f"   Recall:    {detailed_reports[name]['weighted avg']['recall']:.4f}")
    print()

results_df = pd.DataFrame(
    [
        {
            "Model": name,
            "Accuracy": acc,
            "Precision": report['weighted avg']['precision'],
        }
        for name, (acc, report) in zip(
            results.keys(), zip(results.values(), detailed_reports.values())
        )
    ]
).sort_values("Accuracy", ascending=False)

print("🏆 FINAL RESULTS:")
print(results_df.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print(f"\n🎉 Best: {max(results, key=results.get)} ({max(results.values()):.4f})")

📊 Decision Tree: **0.8700** (87.00%)
   Precision: 0.8724
   Recall:    0.8700

📊 Random Forest: **0.8850** (88.50%)
   Precision: 0.8860
   Recall:    0.8850

📊 Logistic Regression: **0.8890** (88.90%)
   Precision: 0.8906
   Recall:    0.8890

📊 Bernoulli NB: **0.8960** (89.60%)
   Precision: 0.8987
   Recall:    0.8960

🏆 FINAL RESULTS:
              Model  Accuracy  Precision
       Bernoulli NB    0.8960     0.8987
Logistic Regression    0.8890     0.8906
      Random Forest    0.8850     0.8860
      Decision Tree    0.8700     0.8724

🎉 Best: Bernoulli NB (0.8960)


In [15]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [8, 10, 12],
    "min_samples_split": [2, 5],
}

rf_tuned = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
rf_tuned.fit(X_train, y_train)

trained_models["Tuned RF"] = rf_tuned.best_estimator_
tuned_pred = rf_tuned.best_estimator_.predict(X_test)
tuned_acc = accuracy_score(y_test, tuned_pred)
results["Tuned RF"] = tuned_acc
detailed_reports["Tuned RF"] = classification_report(
    y_test, tuned_pred, target_names=le.classes_, output_dict=True
)

print(f"🏆 Best RF: {rf_tuned.best_score_:.4f}")
print(f"Params: {rf_tuned.best_params_}")
print(f"Test Accuracy: {tuned_acc:.4f}")

results_df = pd.DataFrame(
    [
        {
            "Model": name,
            "Accuracy": acc,
            "Precision": report['weighted avg']['precision'],
        }
        for name, (acc, report) in zip(
            results.keys(), zip(results.values(), detailed_reports.values())
        )
    ]
).sort_values("Accuracy", ascending=False)

print("🏆 UPDATED RESULTS:")
print(results_df.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print(f"\n🎉 Best: {max(results, key=results.get)} ({max(results.values()):.4f})")

🏆 Best RF: 0.8718
Params: {'max_depth': 8, 'min_samples_split': 5, 'n_estimators': 300}
Test Accuracy: 0.8890
🏆 UPDATED RESULTS:
              Model  Accuracy  Precision
       Bernoulli NB    0.8960     0.8987
           Tuned RF    0.8890     0.8922
Logistic Regression    0.8890     0.8906
      Random Forest    0.8850     0.8860
      Decision Tree    0.8700     0.8724

🎉 Best: Bernoulli NB (0.8960)


In [16]:
best_model_name = max(results, key=results.get)
best_model = trained_models[best_model_name]

joblib.dump(best_model, "disease_model.pkl")
joblib.dump(le, "label_encoder.pkl")

print(f"Model saved successfully: {best_model_name}")
print(f"Saved accuracy: {results[best_model_name]:.4f}")

Model saved successfully: Bernoulli NB
Saved accuracy: 0.8960
